After any interrupted run, restart the runtime before re-running this notebook. Re-running %pip install in a live session produces exactly this stale-import failure.

# Phase 6 v2: reviewed outreach adapter training

Run this notebook on a private Colab GPU. It validates the reviewed v2 manifest before training, masks loss to the assistant response, reports validation loss after every epoch, and saves the adapter only to private Google Drive. It does not expose a model endpoint or send outreach.

In [ ]:
%pip install -q transformers==5.14.1 accelerate==1.14.0
%pip install -q bitsandbytes==0.50.0 peft==0.20.0 safetensors==0.8.0

import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/muzzary/GTM-Agent.git"
REPOSITORY_REF = "codex/phase-6-reviewed-adapter"
REPOSITORY_DIR = Path("/content/GTM-Agent")
if not REPOSITORY_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPOSITORY_REF,
            REPOSITORY_URL,
            str(REPOSITORY_DIR),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIR)],
    check=True,
)
sys.path.insert(0, str(REPOSITORY_DIR))

In [ ]:
import importlib
import importlib.metadata

PINNED_VERSIONS = {
    "transformers": "5.14.1",
    "accelerate": "1.14.0",
    "bitsandbytes": "0.50.0",
    "peft": "0.20.0",
    "safetensors": "0.8.0",
}
RESTART_INSTRUCTION = (
    "Runtime > Restart session, then run all cells top to bottom. "
    "Do not re-run the install cell in a live session."
)
packages_without_version = []
for package_name, pinned_version in PINNED_VERSIONS.items():
    installed_version = importlib.metadata.version(package_name)
    imported_version = getattr(
        importlib.import_module(package_name), "__version__", None
    )
    if installed_version != pinned_version or (
        imported_version is not None and imported_version != installed_version
    ):
        raise RuntimeError(
            f"{package_name} version mismatch: installed={installed_version}, "
            f"imported={imported_version}. {RESTART_INSTRUCTION}"
        )
    if imported_version is None:
        packages_without_version.append(package_name)
print({"packages_without_version": packages_without_version})

In [ ]:
import hashlib
import json
import random
import re
import shutil
from collections import Counter
from datetime import UTC, datetime

import torch
from google.colab import drive
from pydantic import ValidationError

from src.evaluation.phase6_benchmark import load_phase6_benchmark
from src.evaluation.phase6_v2 import (
    build_chat_messages,
    render_v2_prompt,
    training_prompt_input,
    training_target_json,
)
from src.schemas.dataset import DatasetManifestV2
from src.schemas.inference import GroundedOutreachOutput
from src.schemas.training import AdapterArtifactMetadata, TrainingConfigV2
from src.training.dataset import validate_dataset_v2

CONFIG_PATH = REPOSITORY_DIR / "configs/phase6/training-v2.json"
DATASET_PATH = REPOSITORY_DIR / "configs/phase6/dataset-v2.json"
BENCHMARK_PATH = REPOSITORY_DIR / "configs/phase6/benchmark-v2.json"
config_model = TrainingConfigV2.model_validate_json(
    CONFIG_PATH.read_text(encoding="utf-8")
)
config = config_model.model_dump()
dataset = DatasetManifestV2.model_validate_json(
    DATASET_PATH.read_text(encoding="utf-8")
)
benchmark = load_phase6_benchmark(BENCHMARK_PATH)
audit = validate_dataset_v2(dataset, benchmark)
assert audit.passed, audit.errors
assert audit.split_counts["train"] == 100
assert audit.split_counts["validation"] == 24
assert config["max_steps"] == 0, "v2 training must run full epochs"
assert torch.cuda.is_available(), "Phase 6 requires a private Colab GPU runtime."
random.seed(config["seed"])
torch.manual_seed(config["seed"])
drive.mount("/content/drive")
train_examples = [
    example for example in dataset.examples if example.split.value == "train"
]
validation_examples = [
    example
    for example in dataset.examples
    if example.split.value == "validation"
]
assert len(train_examples) == 100
assert len(validation_examples) == 24

ARTIFACT_ROOT = Path("/content/drive/MyDrive/gtm-agent-phase6")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
run_pattern = re.compile(r"^run-(\d{2,})$")
existing_run_numbers = [
    int(match.group(1))
    for path in ARTIFACT_ROOT.iterdir()
    if path.is_dir() and (match := run_pattern.fullmatch(path.name))
]
next_run_number = max(existing_run_numbers, default=0) + 1
RUN_DIR_NAME = f"run-{next_run_number:02d}"
RUN_DIR = ARTIFACT_ROOT / RUN_DIR_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINTS_DIR = RUN_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(exist_ok=False)
(RUN_DIR / "evaluation").mkdir(exist_ok=False)
run_info = {
    "run_dir_name": RUN_DIR_NAME,
    "adapter_id": config["adapter_id"],
    "base_model_id": config["base_model_id"],
    "base_model_revision": config["base_model_revision"],
    "dataset_id": dataset.dataset_id,
    "dataset_version": dataset.dataset_version,
    "epochs": config["epochs"],
    "gradient_accumulation_steps": config["gradient_accumulation_steps"],
    "lora_rank": config["lora_rank"],
    "lora_alpha": config["lora_alpha"],
    "learning_rate": config["learning_rate"],
    "max_length": config["max_length"],
    "seed": config["seed"],
    "train_examples": len(train_examples),
    "validation_examples": len(validation_examples),
}
(RUN_DIR / "run-info.json").write_text(
    json.dumps(run_info, indent=2, sort_keys=True), encoding="utf-8"
)
print({"created_run_directory": str(RUN_DIR)})

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

APPROVED_MODEL_ID = config["base_model_id"]
APPROVED_MODEL_REVISION = config["base_model_revision"]
assert APPROVED_MODEL_ID == "Qwen/Qwen3-4B-Instruct-2507"
assert APPROVED_MODEL_REVISION == "cdbee75f17c01a7cc42f958dc650907174af0554"
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    APPROVED_MODEL_ID,
    revision=APPROVED_MODEL_REVISION,
    trust_remote_code=False,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    APPROVED_MODEL_ID,
    revision=APPROVED_MODEL_REVISION,
    quantization_config=quantization,
    device_map="auto",
    trust_remote_code=False,
    use_safetensors=True,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        r=config["lora_rank"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=config["target_modules"],
    ),
)
model.print_trainable_parameters()

In [ ]:
def encode_example(example):
    prompt = render_v2_prompt(training_prompt_input(example))
    target = training_target_json(example)
    messages = build_chat_messages(prompt, target)
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_length"],
    )
    prompt_only = tokenizer.apply_chat_template(
        build_chat_messages(prompt),
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_length"],
    )
    input_ids = encoded["input_ids"]
    prompt_ids = prompt_only["input_ids"]
    prompt_length = prompt_ids.shape[-1]
    assert prompt_length < input_ids.shape[-1], "assistant span was truncated"
    assert torch.equal(prompt_ids[0], input_ids[0, :prompt_length])
    labels = input_ids.clone()
    labels[:, :prompt_length] = -100
    assert torch.all(labels[:, prompt_length:] != -100)
    encoded["labels"] = labels
    return encoded

parity_example = train_examples[0]
parity_prompt = render_v2_prompt(training_prompt_input(parity_example))
parity_messages = build_chat_messages(parity_prompt)
assert parity_messages == [
    {"role": "system", "content": "Return strict JSON only."},
    {"role": "user", "content": parity_prompt},
]
parity_full = tokenizer.apply_chat_template(
    build_chat_messages(parity_prompt, training_target_json(parity_example)),
    tokenize=True,
    add_generation_prompt=False,
    return_dict=True,
    return_tensors="pt",
)
parity_prompt_only = tokenizer.apply_chat_template(
    parity_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
parity_length = parity_prompt_only["input_ids"].shape[-1]
assert parity_length < parity_full["input_ids"].shape[-1]
assert torch.equal(
    parity_prompt_only["input_ids"][0],
    parity_full["input_ids"][0, :parity_length],
)

In [ ]:
def move_to_model(encoded):
    return {key: value.to(model.device) for key, value in encoded.items()}

@torch.no_grad()
def validation_loss(examples):
    was_training = model.training
    model.eval()
    try:
        losses = []
        for example in examples:
            encoded = move_to_model(encode_example(example))
            losses.append(float(model(**encoded).loss.detach().cpu()))
        return sum(losses) / len(losses)
    finally:
        model.train(was_training)

probe_examples = []
for probe_status, probe_limit in (
    ("drafted", 8),
    ("needs_more_evidence", 2),
    ("disqualified", 1),
    ("opted_out", 1),
):
    probe_examples.extend(
        sorted(
            (
                example
                for example in validation_examples
                if example.approved_output.generation_status == probe_status
            ),
            key=lambda example: example.example_id,
        )[:probe_limit]
    )
assert len(probe_examples) == 12

@torch.inference_mode()
def mode_collapse_probe(examples):
    was_training = model.training
    was_use_cache = model.config.use_cache
    was_gradient_checkpointing = model.is_gradient_checkpointing
    try:
        model.eval()
        model.gradient_checkpointing_disable()
        model.config.use_cache = True
        emitted_status_counts = Counter()
        outputs_with_support_map = 0
        parsed_ok = 0
        status_match = 0
        for example in examples:
            prompt_messages = build_chat_messages(
                render_v2_prompt(training_prompt_input(example))
            )
            encoded = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)
            output_ids = model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=768,
                pad_token_id=tokenizer.eos_token_id,
            )
            generated = output_ids[0, encoded["input_ids"].shape[-1] :]
            raw_output = tokenizer.decode(
                generated, skip_special_tokens=True
            ).strip()
            try:
                parsed = GroundedOutreachOutput.model_validate_json(raw_output)
            except (ValueError, ValidationError):
                continue
            parsed_ok += 1
            emitted_status_counts[parsed.generation_status] += 1
            outputs_with_support_map += bool(parsed.support_map)
            status_match += (
                parsed.generation_status
                == example.approved_output.generation_status
            )
    finally:
        model.config.use_cache = was_use_cache
        if was_gradient_checkpointing:
            model.gradient_checkpointing_enable()
        else:
            model.gradient_checkpointing_disable()
        model.train(was_training)
    print(
        {
            "emitted_status_counts": dict(emitted_status_counts),
            "outputs_with_support_map": outputs_with_support_map,
            "parsed_ok": parsed_ok,
            "status_match": status_match,
        }
    )

def save_adapter(destination, epochs_completed):
    destination.mkdir(parents=True, exist_ok=False)
    model.save_pretrained(destination, safe_serialization=True)
    tokenizer.save_pretrained(destination)
    adapter_files = sorted(
        path
        for path in destination.rglob("*")
        if path.is_file() and path.name != "adapter-metadata.json"
    )
    adapter_hasher = hashlib.sha256()
    for path in adapter_files:
        adapter_hasher.update(
            str(path.relative_to(destination)).encode("utf-8")
        )
        adapter_hasher.update(b"\0")
        adapter_hasher.update(path.read_bytes())
    metadata = {
        "artifact_version": "1.0",
        "adapter_id": config["adapter_id"],
        "adapter_revision": adapter_hasher.hexdigest(),
        "base_model_id": APPROVED_MODEL_ID,
        "base_model_revision": APPROVED_MODEL_REVISION,
        "dataset_id": dataset.dataset_id,
        "dataset_version": dataset.dataset_version,
        "train_examples": len(train_examples),
        "epochs_completed": epochs_completed,
        "trained_steps": steps,
        "created_at": datetime.now(UTC).isoformat(),
    }
    (destination / "adapter-metadata.json").write_text(
        json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8"
    )
    metadata_model = AdapterArtifactMetadata.model_validate(metadata)
    assert metadata_model.adapter_id == config["adapter_id"]
    assert metadata_model.epochs_completed == epochs_completed
    assert metadata_model.train_examples == 100
    return metadata

def retain_recent_checkpoints():
    checkpoint_dirs = sorted(CHECKPOINTS_DIR.glob("epoch-*"))
    for checkpoint_dir in checkpoint_dirs[:-2]:
        shutil.rmtree(checkpoint_dir)

optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
gradient_accumulation_steps = config["gradient_accumulation_steps"]
def accumulation_group_size(batch_index, row_count, accumulation_steps):
    group_start = batch_index - (batch_index % accumulation_steps)
    return min(accumulation_steps, row_count - group_start)

assert accumulation_group_size(9, 10, 4) == 2
steps = 0
model.train()
optimizer.zero_grad(set_to_none=True)
for epoch in range(config["epochs"]):
    model.train()
    train_losses = []
    for batch_index, example in enumerate(train_examples):
        encoded = move_to_model(encode_example(example))
        loss = model(**encoded).loss
        train_losses.append(float(loss.detach().cpu()))
        group_size = accumulation_group_size(
            batch_index, len(train_examples), gradient_accumulation_steps
        )
        (loss / group_size).backward()
        is_update = (batch_index + 1) % gradient_accumulation_steps == 0
        is_last = batch_index + 1 == len(train_examples)
        if is_update or is_last:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            steps += 1
    validation_loss_value = validation_loss(validation_examples)
    mode_collapse_probe(probe_examples)
    checkpoint_dir = CHECKPOINTS_DIR / f"epoch-{epoch + 1:02d}"
    checkpoint_metadata = save_adapter(checkpoint_dir, epoch + 1)
    retain_recent_checkpoints()
    print(
        {
            "epoch": epoch + 1,
            "train_loss": sum(train_losses) / len(train_losses),
            "validation_loss": validation_loss_value,
            "optimizer_steps": steps,
        }
    )
expected_steps = (
    (len(train_examples) + gradient_accumulation_steps - 1)
    // gradient_accumulation_steps
)
assert steps == config["epochs"] * expected_steps
ADAPTER_DIR = RUN_DIR / "adapter"
final_metadata = save_adapter(ADAPTER_DIR, config["epochs"])
print({"completed_run_directory": str(RUN_DIR), "metadata": final_metadata})

In [ ]:
print({"completed_run_directory": str(RUN_DIR), "adapter_directory": str(ADAPTER_DIR)})

## Training handoff

The reviewed manifest is validated before model loading, the loss is restricted to the assistant response span, and train/validation loss is printed after every full epoch. The only persisted training artifacts are the adapter, tokenizer, and metadata under the private Drive directory.

The mode-collapse probe prints a stratified validation sample after each epoch. Watch for the collapse signature: if `outputs_with_support_map` is zero while `parsed_ok` is high, the model has collapsed to abstention-only; stop the run rather than letting it finish.